# Forecast player revenue over 120 days

A user acquisition team wants an early revenue estimate to help decide whether to continue buying traffic. The source assessment contains event logs but no campaign costs or acquisition labels. This notebook uses newly generated synthetic data. It does not contain the source records or reproduce their exact results.

## 1. Generate synthetic event data

The event codes follow the assessment structure: login, purchase, level-up, tutorial completion, and ad revenue. Every synthetic player has a complete 120-day observation window. The values and players are artificial.

In [ ]:
import csv
import math
import random
from collections import defaultdict

random.seed(2026)
N_USERS = 6000
HORIZON_DAYS = 120
events = []

def add_event(user_id, day, event_id, value=0.0):
    seconds = day * 86400 + random.randrange(0, 86400)
    events.append({'id': user_id, 'cc': 'SYNTHETIC', 'eventId': event_id,
                   'eventSar': seconds, 'eventValue': round(value, 4)})

for number in range(N_USERS):
    user_id = f'SYN-{number:06d}'
    engagement = random.uniform(0.65, 1.35)
    ad_monetized = random.random() < 0.15
    activity_days = []
    for day in range(HORIZON_DAYS):
        chance = min(0.85, (0.48 * math.exp(-day / 28) + 0.025) * engagement)
        if day == 0 or random.random() < chance:
            activity_days.append(day)
            add_event(user_id, day, 1)
            if ad_monetized and random.random() < 0.28:
                add_event(user_id, day, 5, random.uniform(0.002, 0.04))
            if random.random() < 0.12:
                add_event(user_id, day, 3, random.randint(1, 10))
    if random.random() < min(0.9, 0.58 * engagement):
        add_event(user_id, random.randint(0, 3), 4, 1)

    payer_type = random.choices(['none', 'early', 'late'], weights=[0.979, 0.012, 0.009])[0]
    if payer_type != 'none' and activity_days:
        first_day = random.choice([d for d in activity_days if d < 8]) if payer_type == 'early' and any(d < 8 for d in activity_days) else None
        if payer_type == 'late':
            late_days = [d for d in activity_days if 8 <= d < HORIZON_DAYS]
            first_day = random.choice(late_days) if late_days else None
        if first_day is not None:
            add_event(user_id, first_day, 2, random.lognormvariate(3.2, 1.0))
            for day in activity_days:
                if day > first_day and random.random() < 0.02:
                    add_event(user_id, day, 2, random.lognormvariate(3.0, 0.9))

events.sort(key=lambda row: (row['id'], row['eventSar'], row['eventId']))
print(f'Synthetic users: {N_USERS:,}')
print(f'Synthetic event rows: {len(events):,}')
print('Event-code counts:', {event_id: sum(row['eventId'] == event_id for row in events) for event_id in range(1, 6)})

## 2. Build early features and the 120-day target

The target is each player's purchase revenue plus ad revenue during the first 120 days, from day 0 up to but not including day 120. Early features use only events from the first 7 days. A real deployment must use a fixed feature cutoff and a confirmed complete target window.

In [ ]:
players = defaultdict(lambda: {'d7_revenue': 0.0, 'revenue_120': 0.0,
                               'active_days_7': set(), 'tutorial': 0, 'max_level_7': 0})
for row in events:
    user = players[row['id']]
    day = row['eventSar'] // 86400
    event_id = row['eventId']
    value = row['eventValue']
    if event_id in (2, 5):
        user['revenue_120'] += value
        if day < 7:
            user['d7_revenue'] += value
    if event_id == 1 and day < 7:
        user['active_days_7'].add(day)
    if event_id == 4 and day < 7:
        user['tutorial'] = 1
    if event_id == 3 and day < 7:
        user['max_level_7'] = max(user['max_level_7'], value)

rows = []
for user_id, user in players.items():
    rows.append({'id': user_id, 'd7_revenue': user['d7_revenue'],
                 'active_days_7': len(user['active_days_7']),
                 'tutorial': user['tutorial'], 'max_level_7': user['max_level_7'],
                 'revenue_120': user['revenue_120']})

mean_120 = sum(row['revenue_120'] for row in rows) / len(rows)
payer_share = sum(row['revenue_120'] > 0 for row in rows) / len(rows)
print(f'Average synthetic 120-day revenue per player: ${mean_120:.2f}')
print(f'Synthetic players with any revenue: {payer_share:.1%}')

## 3. Test a simple early forecast

Use a fixed, reproducible split by synthetic player ID. The forecast predicts the training-set average 120-day revenue for players in the same early-revenue and activity bands. Compare it with a global-average baseline on held-out players. This is a transparent benchmark, not a production-ready model.

In [ ]:
def segment(row):
    revenue_band = 'zero' if row['d7_revenue'] == 0 else ('low' if row['d7_revenue'] < 1 else 'high')
    activity_band = 'low' if row['active_days_7'] < 3 else 'high'
    return revenue_band, activity_band, row['tutorial']

train = [row for row in rows if int(row['id'][-6:]) % 5 != 0]
test = [row for row in rows if int(row['id'][-6:]) % 5 == 0]
segment_totals = defaultdict(lambda: [0.0, 0])
for row in train:
    key = segment(row)
    segment_totals[key][0] += row['revenue_120']
    segment_totals[key][1] += 1
global_mean = sum(row['revenue_120'] for row in train) / len(train)
predicted_mean = {key: total / count for key, (total, count) in segment_totals.items()}

def mae(actual, predicted):
    return sum(abs(a - p) for a, p in zip(actual, predicted)) / len(actual)

actual = [row['revenue_120'] for row in test]
baseline = [global_mean] * len(test)
forecast = [predicted_mean.get(segment(row), global_mean) for row in test]
baseline_mae = mae(actual, baseline)
forecast_mae = mae(actual, forecast)
print(f'Test users: {len(test):,}')
print(f'Global-average baseline MAE: ${baseline_mae:.2f}')
print(f'Segmented forecast MAE: ${forecast_mae:.2f}')
print(f'MAE change vs baseline: {(forecast_mae / baseline_mae - 1):.1%}')

The error metrics describe only this generated sample. They do not measure performance on the assessment data or future marketing cohorts. The comparison also treats player-level errors equally. A marketing decision may need a cohort-level confidence range and a conservative forecast.

## 4. What marketing needs before a stop-or-scale decision

- Join predictions to campaign and channel labels.
- Join the matching cost of acquiring each player.
- Confirm that every training target has a complete 120-day window.
- Backtest on later acquisition cohorts, not a random split alone.
- Report cohort revenue forecasts with uncertainty and payback timing.
- Set a decision threshold and a minimum sample size with marketing.
- Monitor forecast error as new cohorts mature.

## Conclusion

Early behavior can support a forecast of later player revenue. It cannot answer whether a specific campaign is profitable unless the data include acquisition source, cost, and a reliable observation window. The generated data in this notebook demonstrate the workflow only.